In [1]:
import pandas as pd
import numpy as np
from scipy.stats import zscore

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


In [119]:
# Replace with your correct file paths
batting_pp = pd.read_csv('batting_powerplay.csv')
batting_mid = pd.read_csv('batting_middle.csv')
batting_death = pd.read_csv('batting_death.csv')

bowling_pp = pd.read_csv('bowling_powerplay.csv')
bowling_mid = pd.read_csv('bowling_middle.csv')
bowling_death = pd.read_csv('bowling_death.csv')

auctions = pd.read_csv('auctions_final.csv')
# inflation = pd.read_csv('inflation.csv')


In [120]:
# Batting filters
batting_pp = batting_pp[batting_pp['runs'] >= 30]
batting_mid = batting_mid[batting_mid['runs'] >= 40]
batting_death = batting_death[batting_death['runs'] >= 25]

# Bowling filters
bowling_pp = bowling_pp[bowling_pp['balls'] > 5]
bowling_mid = bowling_mid[bowling_mid['balls'] > 5]
bowling_death = bowling_death[bowling_death['balls'] > 5]


In [121]:
bowling_pp.team.value_counts()

team
Sunrisers Hyderabad            152
Mumbai Indians                 151
Royal Challengers Bengaluru    150
Punjab Kings                   149
Delhi Capitals                 148
Kolkata Knight Riders          139
Rajasthan Royals               122
Chennai Super Kings            110
Lucknow Super Giants            90
Gujarat Titans                  71
Name: count, dtype: int64

In [122]:
def normalize_phase(df, metrics, group_cols=['season']):
    df = df.copy()
    for col in metrics:
        df[f'{col}_z'] = df.groupby(group_cols)[col].transform(lambda x: zscore(x, ddof=1) if len(x) > 1 else 0)
    # Shift to positive scale (min-max after z)
    for col in metrics:
        z_col = f'{col}_z'
        df[z_col] = (df[z_col] - df[z_col].min()) / (df[z_col].max() - df[z_col].min() + 1e-9)
    return df


In [123]:
batting_weights = {
    'powerplay': {'runs_z': 0.8, 'strike_rate_z': 0.5, 'boundary_pct_z': 0.45, 'dot_pct_z': -0.3},
    'middle': {'runs_z': 0.8, 'strike_rate_z': 0.3, 'boundary_pct_z': 0.2, 'dot_pct_z': -0.2},
    'death': {'runs_z': 0.8, 'strike_rate_z': 0.5, 'boundary_pct_z': 0.3, 'dot_pct_z': -0.4}
}

bowling_weights = {
    'powerplay': {'wickets_z': 1.5, 'economy_z': -0.5, 'dot_pct_z': 0.3, 'boundary_pct_conceded_z': -0.2},
    'middle': {'wickets_z': 1.5, 'economy_z': -0.8, 'dot_pct_z': 0.3, 'boundary_pct_conceded_z': -0.2},
    'death': {'wickets_z': 1.5, 'economy_z': -0.4, 'dot_pct_z': 0.3, 'boundary_pct_conceded_z': -0.3}
}


In [124]:
def compute_phase_score(df, weights, phase_name, metrics):
    df = normalize_phase(df, metrics)
    df['impact_score'] = sum(df[col] * w for col, w in weights.items())
    min_score = df['impact_score'].min()
    df['impact_score_pos'] = df['impact_score'] + abs(min_score)
    df['phase'] = phase_name
    return df


In [125]:
bat_pp = compute_phase_score(batting_pp, batting_weights['powerplay'], 'powerplay',
                             ['runs', 'strike_rate', 'boundary_pct', 'dot_pct'])
bat_mid = compute_phase_score(batting_mid, batting_weights['middle'], 'middle',
                              ['runs', 'strike_rate', 'boundary_pct', 'dot_pct'])
bat_death = compute_phase_score(batting_death, batting_weights['death'], 'death',
                                ['runs', 'strike_rate', 'boundary_pct', 'dot_pct'])


In [83]:
bat_pp.to_csv("temp.csv")

In [126]:
bowl_pp = compute_phase_score(
    bowling_pp,
    bowling_weights['powerplay'],
    'powerplay',
    ['wickets', 'economy', 'dot_pct', 'boundary_pct_conceded']
)

bowl_mid = compute_phase_score(
    bowling_mid,
    bowling_weights['middle'],
    'middle',
    ['wickets', 'economy', 'dot_pct', 'boundary_pct_conceded']
)

bowl_death = compute_phase_score(
    bowling_death,
    bowling_weights['death'],
    'death',
    ['wickets', 'economy', 'dot_pct', 'boundary_pct_conceded']
)


In [127]:

inflation = pd.DataFrame({
    "year": [2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025],
    "inflation": [2.1229721486,1.9370183837,1.7455333727,1.6363863999,1.5598002096,1.4923461630,
                  1.4442525530,1.3893723453,1.3261165843,1.2437784508,1.1830861323,1.10879675,1.0495]
})


In [128]:
auctions = pd.read_csv("auctions_final.csv")  # or your unified auction file
auctions = auctions.merge(inflation, left_on="year", right_on="year", how="left")
auctions["adj_price"] = auctions["price"] * auctions["inflation"]


In [87]:
df = bat_pp.merge(auctions[["name","year","adj_price"]], 
                      left_on=["player","season"], 
                      right_on=["name","year"], 
                      how="left")

# Calculate ROI (normalize impact if needed)
df["ROI"] = df["impact_score"] / (df["adj_price"]/1e8)  # price in millions
df.to_csv("batsman_roi_powerplay.csv", index=False)


In [88]:
df = bat_mid.merge(auctions[["name","year","adj_price"]], 
                      left_on=["player","season"], 
                      right_on=["name","year"], 
                      how="left")

# Calculate ROI (normalize impact if needed)
df["ROI"] = df["impact_score"] / (df["adj_price"]/1e8)  # price in millions
df.to_csv("batsman_roi_middle.csv", index=False)


In [89]:
df = bat_death.merge(auctions[["name","year","adj_price"]], 
                      left_on=["player","season"], 
                      right_on=["name","year"], 
                      how="left")

# Calculate ROI (normalize impact if needed)
df["ROI"] = df["impact_score"] / (df["adj_price"]/1e8)  # price in millions
df.to_csv("batsman_roi_death.csv", index=False)


In [90]:
df = bowl_pp.merge(auctions[["name","year","adj_price"]], 
                      left_on=["player","season"], 
                      right_on=["name","year"], 
                      how="left")

# Calculate ROI (normalize impact if needed)
df["ROI"] = df["impact_score"] / (df["adj_price"]/1e8)  # price in millions
df.to_csv("bowler_roi_powerplay.csv", index=False)


In [91]:
df = bowl_mid.merge(auctions[["name","year","adj_price"]], 
                      left_on=["player","season"], 
                      right_on=["name","year"], 
                      how="left")

# Calculate ROI (normalize impact if needed)
df["ROI"] = df["impact_score"] / (df["adj_price"]/1e8)  # price in millions
df.to_csv("bowler_roi_middle.csv", index=False)


In [92]:
df = bowl_death.merge(auctions[["name","year","adj_price"]], 
                      left_on=["player","season"], 
                      right_on=["name","year"], 
                      how="left")

# Calculate ROI (normalize impact if needed)
df["ROI"] = df["impact_score"] / (df["adj_price"]/1e8)  # price in millions
df.to_csv("bowler_roi_death.csv", index=False)


In [129]:
def inner_join(dfname, dff):
    df = dff.merge(auctions[["name","year","adj_price"]], 
                      left_on=["player","season"], 
                      right_on=["name","year"], 
                      how="inner")
    
    df["ROI"] = df["impact_score_pos"] / (df["adj_price"]/1e8)  # price in millions
    df.to_csv(f"{dfname}_inner_join.csv", index=False)

In [130]:
bowl_pp.team.value_counts()

team
Sunrisers Hyderabad            152
Mumbai Indians                 151
Royal Challengers Bengaluru    150
Punjab Kings                   149
Delhi Capitals                 148
Kolkata Knight Riders          139
Rajasthan Royals               122
Chennai Super Kings            110
Lucknow Super Giants            90
Gujarat Titans                  71
Name: count, dtype: int64

In [101]:
df = bowl_death.merge(auctions[["name","year","adj_price"]], 
                      left_on=["player","season"], 
                      right_on=["name","year"], 
                      how="inner")

# Calculate ROI (normalize impact if needed)
df["ROI"] = df["impact_score"] / (df["adj_price"]/1e8)  # price in millions
df.to_csv("bowler_roi_death_inner_join.csv", index=False)

In [131]:
for i in [(bowl_pp, "bowl_pp"),
    (bowl_mid, "bowl_mid"),
    (bowl_death, "bowl_death"),
    (bat_death, "bat_death"),
    (bat_mid, "bat_mid"),
    (bat_pp, "bat_pp")]:
    inner_join(i[1], i[0])

In [132]:
bat_mid = pd.read_csv("bat_mid_inner_join.csv")
bowl_mid = pd.read_csv("bowl_mid_inner_join.csv")

# Combine all three phases later; here’s the structure for one:
bat_player_roi = (
    bat_mid.groupby(["season","player","team"])["ROI"]
    .mean()   # or sum() if you want cumulative impact
    .reset_index()
)


In [133]:
franchise_roi = (
    bat_mid.groupby(["season","team"])["ROI"].mean().reset_index()
)
franchise_roi.rename(columns={"ROI":"batting_ROI"}, inplace=True)

bowl_franchise = (
    bowl_mid.groupby(["season","team"])["ROI"].mean().reset_index()
)
bowl_franchise.rename(columns={"ROI":"bowling_ROI"}, inplace=True)

# Combine both sides
team_roi = franchise_roi.merge(bowl_franchise, on=["season","team"], how="outer")
team_roi["overall_ROI"] = team_roi[["batting_ROI","bowling_ROI"]].mean(axis=1)
team_roi.to_csv("franchise_overall_roi.csv", index=False)


In [134]:
# --- File mapping ---
files = {
    "bat_pp": "bat_pp_inner_join.csv",
    "bat_mid": "bat_mid_inner_join.csv",
    "bat_death": "bat_death_inner_join.csv",
    "bowl_pp": "bowl_pp_inner_join.csv",
    "bowl_mid": "bowl_mid_inner_join.csv",
    "bowl_death": "bowl_death_inner_join.csv"
}

# --- Load all dataframes into a dictionary ---
dfs = {k: pd.read_csv(v) for k, v in files.items()}

# --- Helper: aggregate by season + team ---
def team_phase_roi(df):
    return (
        df.groupby(["season", "team"])["ROI"]
        .mean()          # avg ROI per franchise per phase
        .reset_index()
    )

# --- Compute ROI per phase ---
team_rois = {k: team_phase_roi(df) for k, df in dfs.items()}

# --- Merge all six phase ROIs ---
final = (
    team_rois["bat_pp"]
    .merge(team_rois["bat_mid"], on=["season", "team"], how="outer", suffixes=("_bat_pp", "_bat_mid"))
    .merge(team_rois["bat_death"], on=["season", "team"], how="outer")
    .merge(team_rois["bowl_pp"], on=["season", "team"], how="outer", suffixes=("", "_bowl_pp"))
    .merge(team_rois["bowl_mid"], on=["season", "team"], how="outer", suffixes=("", "_bowl_mid"))
    .merge(team_rois["bowl_death"], on=["season", "team"], how="outer", suffixes=("", "_bowl_death"))
)

# --- Rename merged columns for clarity ---
final.columns = [
    "season", "team",
    "ROI_bat_pp", "ROI_bat_mid", "ROI_bat_death",
    "ROI_bowl_pp", "ROI_bowl_mid", "ROI_bowl_death"
]

# --- Calculate aggregate ROIs ---
final["batting_ROI"] = final[["ROI_bat_pp", "ROI_bat_mid", "ROI_bat_death"]].mean(axis=1)
final["bowling_ROI"] = final[["ROI_bowl_pp", "ROI_bowl_mid", "ROI_bowl_death"]].mean(axis=1)
final["overall_ROI"] = final[["batting_ROI", "bowling_ROI"]].mean(axis=1)

# --- Save final outputs ---
final.to_csv("franchise_phasewise_and_overall_ROI.csv", index=False)

print("✅ Aggregated ROI file saved as 'franchise_phasewise_and_overall_ROI.csv'")


✅ Aggregated ROI file saved as 'franchise_phasewise_and_overall_ROI.csv'
